# Phase 4 — RPi 용도 추천 AI: GRPO (강화학습)

**런타임 설정**: `런타임 > 런타임 유형 변경 > 하드웨어 가속기: T4 GPU`

Phase 2 SFT는 모델이 URL을 통째로 암기하게 해서 할루시네이션이 생겼음 -> Phase RAG에서
검색으로 실제 후보를 주는 구조로 전환. 이 노트북은 그 위에서, "주어진 후보 중 가장 적절한
걸 골라서 사실에 충실하게 설명하기"를 GRPO로 강화한다.

- **시작 모델**: `Qwen/Qwen2.5-1.5B-Instruct` 베이스 (Phase 2에서 SFT한 모델이 아니라
  새로 시작 — SFT는 암기 위주 템플릿을 학습해서 이 RAG 태스크에는 오히려 방해가 될 수 있음)
- **Reward**: Phase 3에서 만든 것과 동일한 2단계
  1. 규칙 기반 grounding check — 후보에 없는 URL을 쓰면 API 호출 없이 즉시 -3
  2. 통과하면 DeepSeek API가 정확성/유용성/실용성 채점 (+3~-3)
- **주의**: reward function이 completion마다 DeepSeek API를 호출해서 순수 로컬 RL보다
  많이 느림. 기본값(프롬프트 100개, num_generations=4, epoch=2)이면 완료까지 꽤 걸릴 수
  있으니 처음엔 작게(prompts 슬라이스, epoch=1)로 돌려보는 걸 추천.

In [ ]:
!nvidia-smi

In [ ]:
!pip uninstall -y -q torchao
!pip install -q -U "transformers>=4.46" "trl>=0.12" peft accelerate datasets

## 1. DeepSeek API 키 입력

Colab 왼쪽 사이드바 열쇠 아이콘(Secrets)에 `DEEPSEEK_API_KEY`를 등록해두면 자동으로 읽고, 없으면 직접 입력하라고 물어본다. 노트북 파일 자체에는 키가 저장되지 않는다.

In [ ]:
import os

try:
    from google.colab import userdata
    DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
except Exception:
    DEEPSEEK_API_KEY = None

if not DEEPSEEK_API_KEY:
    from getpass import getpass
    DEEPSEEK_API_KEY = getpass("DEEPSEEK_API_KEY 입력: ")

os.environ["DEEPSEEK_API_KEY"] = DEEPSEEK_API_KEY
print("키 설정 완료 (길이:", len(DEEPSEEK_API_KEY), ")")

## 2. 데이터 다운로드 (GitHub raw)

In [ ]:
import urllib.request, os

os.makedirs("data", exist_ok=True)
BASE = "https://raw.githubusercontent.com/wqrvQ2WR/rpi-ai/main/data"
urllib.request.urlretrieve(f"{BASE}/grpo_prompts.jsonl", "data/grpo_prompts.jsonl")
print("size:", os.path.getsize("data/grpo_prompts.jsonl"), "bytes")

In [ ]:
from datasets import load_dataset

full_ds = load_dataset("json", data_files="data/grpo_prompts.jsonl", split="train")

# 처음엔 작게 테스트해보고 싶으면 아래 슬라이스 범위를 줄이세요 (예: range(30))
N_TRAIN = 100
N_EVAL = 10
train_ds = full_ds.select(range(N_TRAIN))
eval_ds = full_ds.select(range(N_TRAIN, N_TRAIN + N_EVAL))
print(train_ds)
print(train_ds[0]["prompt"])

## 3. 모델 / 토크나이저 로드

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)

## 4. LoRA 설정

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

## 5. Reward function

Phase 3의 `scripts/reward_model.py`와 같은 로직(규칙 기반 grounding check + DeepSeek judge)을
노트북 안에 그대로 옮겨왔다 (검색 관련 의존성 없이 이 노트북만으로 동작하게 하기 위해).

In [ ]:
import json
import re
import urllib.request

DEEPSEEK_URL = "https://api.deepseek.com/chat/completions"
DEEPSEEK_MODEL = "deepseek-chat"

JUDGE_SYSTEM_PROMPT = '''당신은 라즈베리파이 프로젝트 추천 AI의 답변 품질을 채점하는 평가자입니다.
아래 [질문], [실제 후보 목록](이 안의 정보만 사실로 인정), [AI의 답변]을 보고 채점하세요.

점수 기준 (정수 하나만):
+3: 후보 중 가장 적절한 걸 골라서 정확하고 실용적으로 설명함
+1: 대체로 적절하지만 설명이 부실하거나 최선의 선택은 아님
-1: 후보와 동떨어지거나 질문 의도를 잘못 이해함
-3: 후보 목록에 없는 사실을 지어내거나 완전히 틀린 정보를 줌

반드시 JSON만 출력하세요: {"score": <정수>, "reason": "<한 문장 이유, 한국어>"}'''


def grounding_check(answer, candidates):
    candidate_urls = {c["url"] for c in candidates}
    found_urls = set(re.findall(r"https?://[^\s)\]\[]+", answer))
    found_urls = {u.rstrip(".,\uff0c\u3002") for u in found_urls}
    bad_urls = [u for u in found_urls if u not in candidate_urls]
    return (len(bad_urls) == 0, bad_urls)


def call_deepseek(system, user):
    payload = {
        "model": DEEPSEEK_MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "temperature": 0,
        "max_tokens": 200,
    }
    req = urllib.request.Request(
        DEEPSEEK_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}",
        },
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        result = json.loads(resp.read())
    return result["choices"][0]["message"]["content"]


def llm_judge_score(query, candidates, answer):
    candidate_text = "\n".join(
        f"- {c['title']}: {c['description']} (\ub9c1\ud06c: {c['url']})" for c in candidates
    )
    user_prompt = f"[\uc9c8\ubb38]\n{query}\n\n[\uc2e4\uc81c \ud6c4\ubcf4 \ubaa9\ub85d]\n{candidate_text}\n\n[AI\uc758 \ub2f5\ubcc0]\n{answer}"
    try:
        raw = call_deepseek(JUDGE_SYSTEM_PROMPT, user_prompt)
    except Exception as e:
        print("DeepSeek 호출 실패:", e)
        return 0.0
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        return 0.0
    json_text = re.sub(r'"score"\s*:\s*\+', '"score": ', match.group(0))
    try:
        return float(json.loads(json_text)["score"])
    except (json.JSONDecodeError, KeyError, ValueError):
        return 0.0


def _completion_text(c):
    if isinstance(c, list):
        return c[-1]["content"]
    return c


def reward_fn(prompts, completions, candidates, **kwargs):
    rewards = []
    for prompt, completion, cand_json in zip(prompts, completions, candidates):
        cand_list = json.loads(cand_json)
        answer = _completion_text(completion)
        # prompt의 user 메시지에서 [질문] 뒤 텍스트를 질문으로 추출
        user_msg = prompt[-1]["content"] if isinstance(prompt, list) else prompt
        query = user_msg.split("[\uc9c8\ubb38]")[-1].strip()

        ok, bad_urls = grounding_check(answer, cand_list)
        if not ok:
            rewards.append(-3.0)
            continue
        rewards.append(llm_judge_score(query, cand_list, answer))
    return rewards

## 6. 학습 전 베이스라인 평가 (eval set)

In [ ]:
from transformers import pipeline

def evaluate(dataset, model_to_eval):
    pipe = pipeline("text-generation", model=model_to_eval, tokenizer=tokenizer,
                     max_new_tokens=250, do_sample=False)
    scores = []
    for row in dataset:
        out = pipe(row["prompt"])
        answer = out[0]["generated_text"][-1]["content"]
        cand_list = json.loads(row["candidates"])
        query = row["prompt"][-1]["content"].split("[\uc9c8\ubb38]")[-1].strip()
        ok, _ = grounding_check(answer, cand_list)
        score = -3.0 if not ok else llm_judge_score(query, cand_list, answer)
        scores.append(score)
    return sum(scores) / len(scores), scores

baseline_avg, baseline_scores = evaluate(eval_ds, model)
print(f"학습 전 평균 리워드: {baseline_avg:.2f} / 3.0")

## 7. GRPO 학습

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="rpi-ai-qwen2.5-1.5b-grpo",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_generations=4,
    max_completion_length=300,
    learning_rate=1e-5,
    logging_steps=5,
    save_strategy="epoch",
    fp16=True,
    bf16=False,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=grpo_config,
    train_dataset=train_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.train()

## 8. 학습 후 평가 + 저장

In [ ]:
trained_avg, trained_scores = evaluate(eval_ds, trainer.model)
print(f"학습 전: {baseline_avg:.2f} / 3.0")
print(f"학습 후: {trained_avg:.2f} / 3.0")

In [ ]:
trainer.save_model("rpi-ai-qwen2.5-1.5b-grpo/adapter")
tokenizer.save_pretrained("rpi-ai-qwen2.5-1.5b-grpo/adapter")

from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base, "rpi-ai-qwen2.5-1.5b-grpo/adapter")
merged = merged.merge_and_unload()
merged.save_pretrained("rpi-ai-qwen2.5-1.5b-grpo/merged")
tokenizer.save_pretrained("rpi-ai-qwen2.5-1.5b-grpo/merged")

from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copytree("rpi-ai-qwen2.5-1.5b-grpo/merged", "/content/drive/MyDrive/rpi-ai-grpo-merged", dirs_exist_ok=True)
print("드라이브에 저장 완료: MyDrive/rpi-ai-grpo-merged")